# LoRA fine-tune Qwen2.5-Coder-7B-Instruct for word problem -> scene JSON

Runtime: **GPU** (T4 works; L4/A100 are faster and use bf16). Upload `bundle.zip` from the repo's
`model/colab/` when asked. Outputs: `adapter.zip` (LoRA weights) and `outputs.json`
(raw model outputs for the 27 seed questions and a held-out test sample). Score them locally with

```
python -m parser_model.evaluate --model qwen7b-colab --outputs outputs.json --seed data/seed.jsonl
python -m parser_model.evaluate --model qwen7b-colab --outputs outputs.json --seed data/ft/test.jsonl
```

Data is synthetic (generated templates verified against the solver); no textbook text is in it.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets
from google.colab import files
up = files.upload()            # choose bundle.zip
!unzip -o -q bundle.zip -d data && ls data data/*

In [ ]:
import json, math, random, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "Qwen/Qwen2.5-Coder-7B-Instruct"
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16     # T4 has no bf16; fp16 can overflow, watch the loss
print("bf16" if BF16 else "fp16", torch.cuda.get_device_name(0))

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE, device_map={"": 0}, torch_dtype=DTYPE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE))
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

In [ ]:
MAXLEN = 1024

def encode(example):
    msgs = example["messages"]                       # system, user, assistant
    prompt = tok.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    answer = msgs[2]["content"] + "<|im_end|>"
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer, add_special_tokens=False)["input_ids"]
    ids = (p + a)[:MAXLEN]
    labels = ([-100] * len(p) + a)[:MAXLEN]          # loss on the answer only
    return {"input_ids": ids, "labels": labels}

def load(path): return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]
train = [encode(r) for r in load("data/data/ft/train.jsonl")]
val = [encode(r) for r in load("data/data/ft/val.jsonl")][:100]
print(len(train), "train", len(val), "val; longest", max(len(x["input_ids"]) for x in train))

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda xs, v: xs + [v] * (n - len(xs))
    return {
        "input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
        "attention_mask": torch.tensor([pad([1]*len(b["input_ids"]), 0) for b in batch]),
        "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
    }

args = TrainingArguments(
    output_dir="out", num_train_epochs=2, per_device_train_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_steps=20, logging_steps=10, eval_strategy="steps", eval_steps=100,
    per_device_eval_batch_size=4, save_strategy="no", report_to="none",
    bf16=BF16, fp16=not BF16, max_grad_norm=1.0, remove_unused_columns=False,
    gradient_checkpointing=True, optim="paged_adamw_8bit")
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=val, data_collator=collate)
trainer.train()

In [ ]:
model.save_pretrained("adapter"); tok.save_pretrained("adapter")
!zip -q -r adapter.zip adapter

In [ ]:
# Generate raw outputs with greedy decoding. Same prompt as training (the system message is copied
# from the training data, so it cannot drift from the repo).
SYSTEM = load("data/data/ft/train.jsonl")[0]["messages"][0]["content"]
seed = [r for r in load("data/data/seed.jsonl") if not r.get("pending")]
test = load("data/data/ft/test.jsonl")
random.Random(0).shuffle(test)
items = seed + test[:300]                            # raise 300 for a bigger test sample

model.eval(); model.config.use_cache = True; tok.padding_side = "left"
outputs = {}
for i in range(0, len(items), 8):
    chunk = items[i:i+8]
    prompts = [tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},{"role":"user","content":f"Problem: {r['question']}\nJSON:"}],
        tokenize=False, add_generation_prompt=True) for r in chunk]
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=900, do_sample=False, pad_token_id=tok.pad_token_id)
    for r, o in zip(chunk, out):
        outputs[r["id"]] = tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(i + len(chunk), "/", len(items))
json.dump(outputs, open("outputs.json", "w"))
files.download("outputs.json"); files.download("adapter.zip")